<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/homework/WRIT20833_HW3_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 3 — Sentiment Analysis: Tone, Stance, and What Counting Missed

**WRIT 20833 — Intro to Coding in the Humanities**

**Student Name:** _Replace with your name_

**Upload as:** `LASTNAME_HW3.ipynb`

**New to Colab, or submitting?** See [Working in Colab](https://tcu-dcda.github.io/WRIT20833_2026/guides/working-in-colab.html).

---

## What you're doing
This homework runs on **your own dataset**: the clean CSV you made in the Day 12 workshop and
counted in Count Your Data.

- **What counting missed.** In HW2, *commandments* was the top word, but people for the Texas law and
  people against it used it the same way. A word count can't tell a cheer from a complaint.
- **What you add.** VADER, a sentiment tool, scores each comment from very negative (-1) to very
  positive (+1). You'll score every comment in your data and see its overall tone.
- **The question you'll test.** Tone is not stance (Part 4 of the VADER code-along). In **B3** you
  label a few of your comments by hand and check whether VADER's tone could have told you
  which side each writer was on.

## Prepare
Review **our own** materials first:
- **HW2** — term frequency (`split_into_words`, `stopwords`, `Counter`); we reuse all of it.
- **Pandas 01 / Pandas 02** code-alongs — DataFrames, `df[...]`, `.value_counts()`, `df.plot()`.
- The **VADER Sentiment** code-along (this week) — `analyzer.polarity_scores()`, `.apply()` (its
  structure block is in Part 2), what a score of exactly 0 can mean, and the chart.

## Expected: frequent, meaningful `#comments`, plus written reflections
Comment as you work — not every line, but **often, and wherever you make a choice or hit a
question**: say **what you tried** and **what you learned** (or what surprised you). A `#comment`
belongs next to the code it explains: a prediction, a choice, a quick answer.

Longer thinking goes in the **reflection cells**: text cells with the question at the top. Double-click one,
write your answer under the line in plain sentences (no `#` needed), then press **Shift+Enter** to
see it formatted. Under this course's **ungrading** approach, your thinking in these comments and
reflections matters more than getting a "right" number.

## Load your data (two plumbing cells)
**Before you go on, check the `DATA SOURCE:` line the second cell prints.** It should name your
CSV. If it says "THIS IS NOT YOUR DATA," go back and upload your file. Everything after this point
is about whatever data loaded here.

1. **Run the first cell.** A **Choose Files** button appears under it. Click it and pick your clean
   CSV, the same file you used in Count Your Data (it ends in `.csv`, not `_info.json`).
2. **Run the second cell.** It installs VADER and loads your file into a table. You don't need to
   read every line. You *use* a tool through its **interface** long before you build its
   **insides**.

After both run, you have: a DataFrame **`df`** (one comment per row), the name of its text column
**`text_column`**, the **`analyzer`** (VADER), and your HW2 **`split_into_words`** + **`stopwords`**.

**Colab forgets uploaded files** when your session ends. When you come back, run both cells again.

**No button?** Some browsers block it. Upload with the **Files** icon (a folder) in the left
sidebar instead. Then, in the first cell, put your file's name between the quotes in
`MY_FILE = ""`, like `MY_FILE = "youtube_comments_abc123.csv"`.

In [ ]:
# PLUMBING: click "Choose Files" and pick your clean CSV (the same one you used in Count Your Data)
MY_FILE = ""   # No button? Upload with the Files icon instead, then type the file name between these quotes.

if MY_FILE == "":
    try:
        from google.colab import files
        uploaded = files.upload()      # waits for you to choose a file; returns {file name: contents}
    except ImportError:                # not in Colab: no upload button
        uploaded = {}
    if len(uploaded) > 0:
        MY_FILE = list(uploaded.keys())[0]
        if len(uploaded) > 1:
            print(f"Warning: You chose {len(uploaded)} files; this notebook uses only the first one.")
        print("Uploaded:", MY_FILE)
    else:
        print("No file was chosen. Run this cell again and pick your clean CSV.")

In [ ]:
# PLUMBING: SETUP — run this cell next. (You're not expected to read every line yet — that's fine.)
import re, os, urllib.request
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

# VADER is a ready-made sentiment tool. It is not built into Python, so we install it,
# then make one analyzer we reuse. (More on "borrowing" tools in the note after A5.)
!pip install -q vaderSentiment==3.3.2
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

# Same word-splitter + stopwords you used in HW2 (we reuse our own tools).
def split_into_words(any_chunk_of_text):
    return re.split(r"\W+", str(any_chunk_of_text).lower())
stopwords = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now", "ve", "ll", "amp"]

def load_course_comments():
    """Load the course YouTube-comments corpus into a DataFrame, one line of comment text per row."""
    filename = "ten_commandments/tc_youtube_comments.txt"
    text = None
    for path in (filename, os.path.join("notebooks", "data", filename),
                 os.path.join("data", filename), os.path.join("..", "data", filename)):
        if os.path.exists(path):
            text = open(path, encoding="utf-8").read(); break
    if text is None:
        url = "https://raw.githubusercontent.com/TCU-DCDA/WRIT20833_2026/main/notebooks/data/" + filename
        text = urllib.request.urlopen(url).read().decode("utf-8")
    rows = [line.strip() for line in text.split("\n") if line.strip()]
    return pd.DataFrame({"comment": rows})

# ----- Load YOUR data -----------------------------------------------------------
MY_TEXT_COLUMN = ""   # leave blank to find it automatically ("comment_clean" or "text_clean")

if MY_FILE:
    MY_FILE = MY_FILE.strip()
    # keep_default_na=False keeps a comment that says "NA" as text instead of an empty cell
    df = pd.read_csv(MY_FILE, keep_default_na=False)
    text_column = MY_TEXT_COLUMN
    if text_column == "":
        for candidate in ["comment_clean", "text_clean"]:
            if candidate in df.columns:
                text_column = candidate
                break
    if text_column not in df.columns:
        raise ValueError(f"Can't tell which column holds your text. Your columns are {list(df.columns)}. "
                         "Type the right one between the quotes in MY_TEXT_COLUMN above and run this cell again.")
    df[text_column] = df[text_column].astype(str)
    blank = df[text_column].str.strip() == ""
    df = df[~blank].reset_index(drop=True)
    print("DATA SOURCE:", MY_FILE, f"({blank.sum()} blank rows dropped)")
    if len(df) == 0:
        raise ValueError(f"Your file loaded, but its '{text_column}' column has no text in it, so there is "
                         "nothing to analyze. Check that you picked your clean CSV (not the _info.json file "
                         "or a raw export). If it is the right file, bring it to the work session.")
else:
    df = load_course_comments()
    text_column = "comment"
    print("DATA SOURCE: course corpus (Ten Commandments comments)")
    print()
    print("*" * 70)
    print("WARNING: THIS IS NOT YOUR DATA.")
    print("No file was uploaded, so you are looking at the course's backup comments.")
    print("HW3 asks you to work on YOUR OWN dataset. Run the upload cell above,")
    print("pick your clean CSV, then run this cell again.")
    print("*" * 70)
    print()

print(f"{len(df)} rows. Text column: '{text_column}'.")
print("First one:", df[text_column].iloc[0][:120])
df.head()

## Part A — From text to a sentiment score (5 exercises)

**A1 — Meet your data as a table.** *(HW2 was one big blob of text; now each row is one comment.)*

In [ ]:
# A1
# TODO: print df.shape (rows, columns)
# TODO: print how many rows there are with len(df)
# TODO: show the first few rows with df.head()

# #comments:
# Each ROW is now one comment. How is a DataFrame different from HW2's flat word list?

**A2 — Clean text, but keep the feeling.** HW2 *dropped* punctuation and capitals — sentiment tools need them.

In [ ]:
# A2
sample = "I LOVE this!!! Best thing I've seen all year."
print("Original:   ", sample)
# TODO: print sample.lower() and notice what intensity would be LOST (CAPS, "!") if we
#       cleaned it the HW2 way.
# TODO (optional): find a comment in YOUR data with CAPS or "!" and try it here.

# #comments:
# Why does lowercasing + stripping "!" HELP a word count (HW2) but HURT a sentiment score?

**A3 — Score a single comment with VADER.**

In [ ]:
# A3
text = "What a great video, thank you!"
scores = analyzer.polarity_scores(text)
print(scores)

# TODO: print just the 'compound' score (one number, -1 = most negative ... +1 = most positive)
# TODO: change `text` to something you might write, and re-run

# #comments:
# What do 'pos', 'neu', 'neg', and 'compound' each seem to measure?

**A4 — Check your prediction.** In Count Your Data (Prediction 3) you picked one comment
from your data that you expected VADER to misread. Now find out.

Paste that comment between the triple quotes `"""` below. Triple quotes still make a string, and
they let your comment contain quotation marks of its own. No prediction saved? Pick any comment from
your data that you think VADER will get wrong.

In [ ]:
# A4 — predict, then run
tricky = """Oh great, another Monday."""      # replace this example with your Prediction 3 comment

# FIRST, in a #comment: will VADER score it positive or negative, and what will it miss?
# My prediction:

# THEN run the cell.
print(analyzer.polarity_scores(tricky)["compound"])

**A4 reflection** *(double-click to edit; write your answer below the line)*

Did VADER misread your comment the way you predicted? What did it miss (sarcasm, a quote of
the other side, a joke, an emoji, slang)? What does that warn you about trusting an automated score
on its own?

---

_Your answer:_

**A5 — Score every comment at once.** *(This is what pandas is for.)*

In [ ]:
# A5
def get_sentiment(text):
    return analyzer.polarity_scores(text)["compound"]

# TODO: make a new column named "sentiment". Type this whole line, including the df["sentiment"] = part:
#       df["sentiment"] = df[text_column].apply(get_sentiment)
#       (B1, B3, and C1 all use this column. If one of them says KeyError: 'sentiment', come back here.)
# TODO: print df["sentiment"].mean(), .min(), .max()
# TODO: df.head() to see the new column

# #comments:
# In HW2 you looped by hand; here .apply() runs your function on every row.
# What just happened to all the comments in one line?

### A note: you didn't build VADER — and that's normal
VADER is a **sentiment lexicon**: thousands of words hand-scored for feeling, plus rules for
"!", ALL CAPS, and negation, compiled and tuned by researchers. Almost nobody writes one from
scratch — you `pip install` it. (Long before AI, you'd still have reached for a library like
this; today you might ask an AI to help use it.) **Borrowing it is expected, not cheating.**

But A4 showed the catch: a borrowed tool carries someone else's assumptions and can be
**confidently wrong** (sarcasm, jokes, and quoting the other side all fool it). So the real skill isn't *building* VADER — it's
**judging whether to trust it** on your text. That's exactly what the human-vs-VADER check in
**B3** is for. *(This is the thread from **Day 9**, reading and improving AI-written code.)*

## Part B — Tone, stance, and what counting missed (4 exercises)

**B1 — Turn scores into labels.** A number becomes a category. *(Recall Classification Logic.)*

When a row scores exactly 0, you can find out why: search its words in the
[VADER Lexicon Explorer](https://tcu-dcda.github.io/WRIT20833_2026/guides/vader-lexicon.html) on the course
site. A word that isn't listed adds nothing to the score.

In [ ]:
# B1
def label_sentiment(score):
    # Standard VADER cutoffs: >= 0.05 is positive, <= -0.05 is negative, otherwise neutral.
    # TODO: return "positive", "negative", or "neutral" based on score
    pass

# TODO: make df["label"] by applying label_sentiment to df["sentiment"]
#       (B2, B3, and C1 use this column. If `pass` is still above, every label comes out None.)
# TODO: print df["label"].value_counts()
# TODO: print how many rows score EXACTLY 0:  (df["sentiment"] == 0).sum()
#       (== 0 marks each row True or False; .sum() counts the Trues, the same trick as
#       .isnull().sum() in Pandas 02)

# #comments:
# WHO chose the 0.05 cutoff? How would moving it change how many comments count as "positive"?
# How many of your "neutral" rows are exactly 0? A 0 can mean VADER found none of the comment's
# words on its list ("no evidence"), or that its positive and negative words cancelled out. Look up
# one or two in the Lexicon Explorer before you decide which kind yours are.

**B2 — Picture the split.** One line of pandas makes a chart. *(You saw `df.plot()` in the code-alongs.)*

In [ ]:
# B2
# TODO: make a bar chart of the label counts:
#       df["label"].value_counts().plot(kind="bar", title="Sentiment of the comments")

# #comments:
# Does the chart show a clear majority, or a divided crowd?

**B3 — Read the extremes, and label their stance yourself.** *(The human-vs-machine check.)*

In the code-along, every comment came with a `stance` someone labeled by hand. Your data doesn't
have one, so **you** are the labeler. Part 1 picks the 5 comments VADER scored highest and the 5 it
scored lowest, and prints them **without their scores** and in mixed order. Read each one and decide:
is the writer on one side (**support**), the other (**oppose**), or is it **unclear**? Write your
labels down *before* you run part 2. Part 2 reveals VADER's scores, so you can't be swayed by them.

**First, name your two sides.** What are your comments arguing about, and what would "support" and
"oppose" mean? You'll write this at the top of part 2. If your comments aren't really arguing about
anything, you can use whether the writer **approves or disapproves of the video (or its creator)**.
If even that doesn't fit a comment (a question, a joke, advice), label it **"n/a"** and say why in
your reflection.

**Small or one-sided data?** If your file has fewer than 10 comments, the groups will be smaller. If
your data has only a few strongly positive or negative comments, a group may include comments that
score 0. Neither is a bug: both are findings about your data.

*Want a fuller check?* [Checking the Machine](https://github.com/TCU-DCDA/WRIT20833_2026/blob/main/materials/Checking_the_Machine.md) is a one-page routine for this kind of
human-vs-tool comparison. You'll reuse it in HW4 and the capstone.

**What B3 part 1 does, step by step.** The numbers match the `# 1` to `# 6` tags in the code cell.

1. `min(5, len(df) // 2)`: 5 comments from each end, or half your rows if you have fewer than 10, so
   both groups are the same size. (`//` divides and drops the remainder: `7 // 2` is 3.)
2. `.nlargest(how_many, "sentiment")`: keeps the rows with the highest scores.
3. `.drop(highest.index)`: sets those rows aside first, so `.nsmallest` can't pick any of them again.
   Without it, a small or one-sided dataset could put the same comment in both groups.
4. `.assign(group=...)` adds a column saying which group each row came from; `pd.concat([...])` stacks
   the two small tables into one.
5. `.sort_index()` puts the rows in row-number order, which mixes the two groups together;
   `.iterrows()` hands you the table one row at a time, as a pair: the row's number and the row itself.
6. `textwrap.fill(...)` breaks a long comment into lines of at most 90 characters, indented 4 spaces,
   so you can read it without scrolling sideways. (`import textwrap` at the top loads that tool; it
   comes with Python.)

In [ ]:
# B3 — part 1: read the extremes (scores hidden). Steps 1-6 are explained in the text above.
# Run this cell, then read the comments it prints.
import textwrap
how_many = min(5, len(df) // 2)                                     # 1
highest = df.nlargest(how_many, "sentiment")                        # 2
lowest = df.drop(highest.index).nsmallest(how_many, "sentiment")    # 3
extremes = pd.concat([highest.assign(group="highest"),
                      lowest.assign(group="lowest")])               # 4
print(len(highest), "highest-scoring and", len(lowest), "lowest-scoring comments, in row order:")
print()
for row_number, row in extremes.sort_index().iterrows():            # 5
    print(f"[{row_number}]")
    print(textwrap.fill(row[text_column], width=90,
                        initial_indent="    ", subsequent_indent="    "))   # 6
    print()

**How part 2 works.** You type your labels into a dictionary, and pandas attaches them to the table.

**The structure:**
```python
my_stance = {
    row_number: "support",
    another_row_number: "oppose",
}
extremes["stance"] = extremes.index.map(my_stance)
```
- Each **key** is a row number from part 1's printout (the number in square brackets), typed without
  quotes. Each **value** is your label, in quotes.
- `extremes.index` is the table's row numbers. `.map(my_stance)` looks each one up in your dictionary
  and gives back your label, like `my_stance[row_number]` done for every row at once.
- A row you haven't labeled yet shows `NaN` (pandas' mark for an empty cell), so you can see which
  ones are left.
- The two filled-in entries (`0` and `1`) only show the pattern. Replace them with your own row numbers.

In [ ]:
# B3 — part 2: your stance labels
# My comments are arguing about:
# "support" means:
# "oppose" means:

# TODO: for each [row number] above, write "support", "oppose", "unclear", or "n/a".
#       The row numbers on your screen are the keys. (Two are filled in as a pattern -- replace them.)
#       Label every comment BEFORE you run this cell: running it reveals VADER's scores.
my_stance = {
    0: "support",
    1: "oppose",
}

extremes["stance"] = extremes.index.map(my_stance)
extremes[["group", "sentiment", "label", "stance"]]   # the "highest" group first, then the "lowest"

**B3 reflection** *(double-click to edit; write your answer below the line)*

- In the **highest** group, how many did you label "support"? In the **lowest** group? (Use the
  `group` column. If a group has fewer than 5 rows, say how many it has.)
- If positive tone meant support, what would you expect, and what did you find?
- Before you saw the scores, did any comment seem to you to have a different tone than VADER gave
  it? Which one, and why?
- Which labels were hard to decide, and why? (That difficulty is data too.)

---

_Your answer:_

**B4 — Interpretation (write 4–6 sentences in the cell below).** Answer these, in any order:

1. **Tone:** does VADER label your comments mostly positive, mostly negative, or split? (B1 counts,
   B2 chart) Does your own reading of the comments agree?
2. **Neutral:** how many of your "neutral" comments scored exactly 0 (B1)? A 0 can mean VADER found
   no words it knew, or that positive and negative words cancelled out. It does not mean the comment
   was calm. Say what your neutral comments really are, from the ones you checked.
3. **Tone vs. stance:** could you have read your writers' sides off their tone? (your B3 reflection)
4. **Gained and missed:** what did sentiment let you see that HW2's word-counting couldn't, and
   what can't it see?

Cite at least two numbers from this notebook: a label count, a sentiment score, or a count from
your B3 labels.

**One drafting rule.** Politicians of every party love the sentence *"The American people
feel…"*: one average pretending a split crowd agreed. If your write-up starts *"the commenters
feel…"*, check it against your own B1 counts and B2 chart. If the crowd is split, **say split.**

**B4 interpretation** *(double-click to edit; write your answer below the line)*

Your 4–6 sentences, citing at least two numbers.

---

_Your answer:_

## Part C — Going deeper (2 exercises)

**C1 — Whose words are whose? Frequency *inside* each camp.** *(HW2's tool, now split by tone.)*

In [ ]:
# C1
def top_words(comments, how_many=8):   # how_many=8: if you don't say how many, you get 8
    words = []
    for comment in comments:
        for word in split_into_words(comment):
            if word and word not in stopwords:   # skip empty strings and stopwords, as in HW2
                words.append(word)
    return Counter(words).most_common(how_many)

positive_comments = df[df["label"] == "positive"][text_column]
negative_comments = df[df["label"] == "negative"][text_column]

# TODO: print top_words(positive_comments) and top_words(negative_comments)

**C1 reflection** *(double-click to edit; write your answer below the line)*

Which words appear in **both** lists? Is your top word from Count Your Data one of them? If a
word shows up in positive *and* negative comments, does a positive-scoring comment that uses it mean
the writer is on the "support" side? (Check your B3 labels before you answer.)

---

_Your answer:_

**C2 — Push your data further.** Try ONE of these:

- **Compare a word.** Pick a word that matters in your data. Compare the average sentiment of
  comments that mention it with comments that don't. `.str.contains()` from Pandas 02 finds them:
  `df[df[text_column].str.contains("word", case=False)]` keeps the comments that mention it, and a
  `~` in front of the condition flips it, `df[~df[text_column].str.contains("word", case=False)]`,
  to keep the ones that don't. Watch out: `.str.contains("art")` also matches "party" and "start".
- **Move the cutoff.** Copy `label_sentiment` from B1 into this cell under a new name (say,
  `label_strict`) and change its `0.05` cutoffs. Then apply it to the scores,
  `df["sentiment"].apply(label_strict)`, and compare that result's `.value_counts()` with B1's. Leave
  B1 itself alone: B2, B3, and C1 use its labels.

In [ ]:
# C2
# TODO: do ONE of the options above. Show your code and the result.

**C2 reflection** *(double-click to edit; write your answer below the line)*

What did you expect, and did your data agree? Name one place in your data where you doubt
VADER, and say why.

---

_Your answer:_

## Weekly Experiments (your own work!)
C2 already counts as one experiment. After the required exercises, create **1–2 more small
experiments** of your own with this week's skills. Make them original.

Ideas:
- Write five comments yourself and see if VADER scores them the way you meant.
- Find a comment where VADER is **most wrong** and explain why.
- Filter `df` to a subset (comments mentioning a word you care about) and compare its average sentiment.

It's fine if an experiment doesn't work at first — **documenting what went wrong and what you
learned is worth just as much as working code.**

In [ ]:
# Experiment 1
# (#comments in your code as you go)

**Experiment 1 reflection** *(double-click to edit; write your answer below the line)*

What were you trying? What happened? What did you learn?

---

_Your answer:_

In [ ]:
# Experiment 2 (optional)
# (#comments in your code as you go)

**Experiment 2 reflection** *(double-click to edit; write your answer below the line)*

What were you trying? What happened? What did you learn?

---

_Your answer:_

## Submit
- [ ] My name is at the top.
- [ ] I uploaded **my own** clean CSV, and the `DATA SOURCE:` line names my file (not "THIS IS NOT YOUR DATA"). If I couldn't get my file to load, I wrote a `#comment` in the setup cell saying what went wrong. I ran every cell. Where something still breaks, I left a `#comment` about what I tried and what I learned — errors are part of the work here, not something to hide or delete.
- [ ] I completed **Part A (5)**, **Part B (4, incl. the B4 write-up)**, **Part C (2)**, and **1–2 experiments** of my own (C2 already counts as one).
- [ ] I used `#comments` frequently and meaningfully — showing my thinking wherever I made a choice or hit a question (not every line, but throughout).
- [ ] I answered every reflection in its text cell, under the line.
- [ ] I wrote honestly about where I **agree and disagree** with VADER (engagement matters more than "right" labels).
- [ ] Saved as `LASTNAME_HW3.ipynb` and uploaded to its folder in TCU Online → Assignments.